# NexaFlow — Análisis con Apache Spark

Procesa los datasets de ventas y reservas de NexaFlow y genera 5 métricas de negocio.

| # | Métrica |
|---|---|
| 1 | Revenue y ventas por tenant |
| 2 | Top productos por cantidad vendida |
| 3 | Tasa de cancelación de reservas por tenant |
| 4 | Reservas por franja horaria |
| 5 | Ventas completadas por mes |

> **Requisito:** los archivos `sales.csv` y `reservations.csv` deben estar en la misma carpeta que este notebook.

## 0 · Instalación de dependencias

In [ ]:
import importlib, subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for _pkg in ('pyspark',):
    if importlib.util.find_spec(_pkg) is None:
        print(f'Instalando {_pkg}...')
        _install(_pkg)

print('Dependencias listas.')

## 1 · Inicializar Spark y cargar datos

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("NexaFlow-Analytics")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

sales        = spark.read.csv("sales.csv",        header=True, inferSchema=True)
reservations = spark.read.csv("reservations.csv", header=True, inferSchema=True)

print(f"Dataset: {sales.count()} ventas | {reservations.count()} reservas")
sales.printSchema()

## Métrica 1 · Revenue y ventas por tenant

In [ ]:
(sales.filter(F.col("status") == "completed")
      .groupBy("tenant_id")
      .agg(
          F.count("sale_id").alias("total_ventas"),
          F.round(F.sum("total"), 2).alias("revenue_total"),
          F.round(F.avg("total"), 2).alias("ticket_promedio"),
      )
      .orderBy(F.desc("revenue_total"))
      .show(truncate=False))

## Métrica 2 · Top productos por cantidad vendida

In [ ]:
(sales.filter(F.col("status") == "completed")
      .groupBy("product")
      .agg(
          F.sum("quantity").alias("unidades_vendidas"),
          F.round(F.sum("total"), 2).alias("revenue"),
      )
      .orderBy(F.desc("unidades_vendidas"))
      .show(truncate=False))

## Métrica 3 · Tasa de cancelación de reservas por tenant

In [ ]:
total_res = reservations.groupBy("tenant_id").agg(F.count("*").alias("total"))
cancelled = (reservations.filter(F.col("status") == "cancelled")
             .groupBy("tenant_id")
             .agg(F.count("*").alias("canceladas")))

(total_res.join(cancelled, "tenant_id", "left")
          .fillna(0, subset=["canceladas"])
          .withColumn("tasa_cancelacion_%",
                      F.round(F.col("canceladas") / F.col("total") * 100, 1))
          .orderBy(F.desc("tasa_cancelacion_%"))
          .show(truncate=False))

## Métrica 4 · Reservas por franja horaria

In [ ]:
(reservations
      .withColumn("hora", F.split(F.col("time_slot"), ":")[0].cast("int"))
      .withColumn("franja",
          F.when(F.col("hora") < 12, "Mañana (8-12)")
           .when(F.col("hora") < 17, "Tarde (12-17)")
           .otherwise("Noche (17-22)"))
      .groupBy("franja")
      .agg(F.count("*").alias("reservas"))
      .orderBy("franja")
      .show(truncate=False))

## Métrica 5 · Ventas completadas por mes

In [ ]:
(sales.filter(F.col("status") == "completed")
      .withColumn("mes", F.date_format(F.col("created_at"), "yyyy-MM"))
      .groupBy("mes")
      .agg(
          F.count("sale_id").alias("ventas"),
          F.round(F.sum("total"), 2).alias("revenue"),
      )
      .orderBy("mes")
      .show(truncate=False))

spark.stop()
print("Análisis completado.")